# Part 10 — Bybit with real order-book depth (Paper 2)
Part 7 tested Bybit with trades only and failed. This notebook tests the thin-book result on Bybit **with order-book depth**, on days that Parts 7 and 9 never used, against the full Part 9 baseline. Both results go in the paper.

**Order of work**
1. Run **V0, V1, V2**. (If V2 already ran once, it only re-checks the lock.)
2. Run **V3A**. It downloads every coin-day from Bybit's file server by itself (about 1–2 hours). Skip V3 unless V3A says the server is unreachable.
3. Run **V4** (trades, about 15–30 min) and **V5**.
4. Run **V6 exactly once**. It downloads `p10_results.zip`; send that to Claude.

If Colab disconnects, run V0, V1, V2 again and rerun the cell you were on; finished days are kept in Drive.

In [ ]:
# ============================================================================
# CELL V0 — SETUP (self-contained; run first, every time you open the notebook)
# Part 10 (Paper 2): Bybit with REAL ORDER-BOOK DEPTH. Bybit's historical order-book files (ob200: a full-day snapshot
# plus every update to the top 200 levels), downloaded by hand from bybit.com/derivatives/en/history-data
# (Product Line: Contract, Data category: OB Data) and uploaded here. Trades come from Bybit's free archive
# (public.bybit.com, already used in Part 7). Same statistic and same baseline as Part 9.
# Part 7 tested Bybit with trades only and FAILED; this test is reported next to it, never instead of it.
# ============================================================================
import os, sys, json, time, math, glob, re, io, zipfile, hashlib, subprocess, warnings, gzip, shutil
import urllib.request, urllib.error
from concurrent.futures import ThreadPoolExecutor, as_completed

IN_COLAB = "google.colab" in sys.modules
try:
    import pyarrow  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyarrow"], check=True)
import numpy as np
import pandas as pd
from scipy import stats, optimize
warnings.filterwarnings("ignore", category=RuntimeWarning)

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = "/content/drive/MyDrive/ceiling_suite"
    UPLOAD_DIRS = ["/content/bybit_ob", os.path.join(BASE, "bybit_ob")]     # put the .zip files in either folder
else:
    BASE = os.path.abspath("ceiling_suite_local")
    UPLOAD_DIRS = [os.path.abspath("bybit_ob")]
RESULTS, PREREG = os.path.join(BASE, "results"), os.path.join(BASE, "prereg")
OBMIN = os.path.join(BASE, "data_bybit_ob", "book_minute")      # small per-day summaries (about 100 KB each)
TRMIN = os.path.join(BASE, "data_bybit_ob", "trade_minute")
for _d in [RESULTS, PREREG, OBMIN, TRMIN] + UPLOAD_DIRS:
    os.makedirs(_d, exist_ok=True)

SEED = 20261026
def rng(offset=0):
    return np.random.default_rng(SEED + offset)

def _clean(o):
    if isinstance(o, dict):
        return {str(k): _clean(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)):
        return [_clean(v) for v in o]
    if isinstance(o, np.ndarray):
        return _clean(o.tolist())
    if isinstance(o, (np.floating, float)):
        return None if not np.isfinite(o) else float(o)
    if isinstance(o, np.integer):
        return int(o)
    if isinstance(o, np.bool_):
        return bool(o)
    return o

def record(test_id, passed, kind, metrics, criterion="", note=""):
    assert kind in {"theorem-check", "simulation", "real-data", "audit"}, kind
    status = "REPORT" if passed is None else ("PASS" if passed else "FAIL")
    out = {"test_id": test_id, "status": status, "kind": kind, "criterion": criterion, "note": note,
           "metrics": _clean(metrics), "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
           "seed": SEED}
    with open(os.path.join(RESULTS, f"{test_id}.json"), "w") as f:
        json.dump(out, f, indent=2)
    print(f"[{status}] {test_id} ({kind})" + (f" — {criterion}" if criterion else ""))
    if note:
        print("        note:", note)
    return out

# ---- Design (fixed before any test-day file is opened) -------------------------------------------
COINS = ["ACH", "ALICE", "API3", "ARPA", "CELR", "CHR", "CVX", "GTC", "HOT", "ICX", "IOST", "IOTX", "LIT", "OGN", "QNT",
         "RLC", "SPELL", "T", "WOO", "ZEN"]                      # the 20 Binance thin-set coins of Parts 6 and 9
BATCHES = {"A": COINS[0:5], "B": COINS[5:10], "C": COINS[10:15], "D": COINS[15:20]}
# Days 11-16 of a month: never a test day in Part 7 (3, 10, 17, 24) or Part 9 (6, 20); contiguous so that one
# Bybit date-range download covers a whole window.
def _win(y, m):
    return [f"{y}-{m:02d}-{d:02d}" for d in range(11, 17)]
TRAIN_WINDOWS = [(2024, 3), (2024, 9)]
TEST_WINDOWS = [(2025, 1), (2025, 5), (2025, 9), (2026, 1), (2026, 5), (2026, 9)]
TRAIN_D = [d for y, m in TRAIN_WINDOWS for d in _win(y, m)]
TEST_D = [d for y, m in TEST_WINDOWS for d in _win(y, m)]
TRAIN_END = "2024-12-31"
THREADS = 8
print("BASE =", BASE, "| train days", len(TRAIN_D), "| test days", len(TEST_D))

BYBIT = "https://public.bybit.com/trading"
def _http(url, method="GET", timeout=120, tries=3):
    last = None
    for i in range(tries):
        try:
            req = urllib.request.Request(url, method=method, headers={"User-Agent": "Mozilla/5.0"})
            with urllib.request.urlopen(req, timeout=timeout) as r:
                return r.status, (r.read() if method == "GET" else b"")
        except urllib.error.HTTPError as e:
            if e.code in (403, 404):
                return e.code, b""
            last = e
        except Exception as e:
            last = e
        time.sleep(1.5 * (i + 1))
    raise RuntimeError(f"network error {url}: {last}")
def bb_url(sym, date):
    return f"{BYBIT}/{sym}/{sym}{date}.csv.gz"
code, _ = _http(bb_url("BTCUSDT", "2024-06-10"), "HEAD", timeout=30)
print("Bybit trade archive reachable:", code == 200, f"(HTTP {code})")
assert code == 200, "public.bybit.com is not reachable from this runtime; tell Claude."
DL_LOG = []

In [ ]:
# ============================================================================
# CELL V1 — FUNCTIONS (estimation code identical to Parts 6 and 9; Bybit readers added at the end)
# ============================================================================
def prep(df, flow_scale):
    p = np.log(df["close"].to_numpy()) * 1e4
    u = df["flow"].to_numpy() / flow_scale
    tot = (df["bid1"] + df["ask1"]).to_numpy()
    d = ((df["bid1"] - df["ask1"]) / (df["bid1"] + df["ask1"])).to_numpy()
    x = p - pd.Series(p).ewm(span=60, adjust=False).mean().to_numpy()
    return p, u, d, x, tot
def ols(y, X):
    ok = np.all(np.isfinite(X), 1) & np.isfinite(y)
    if ok.sum() < X.shape[1] + 20:
        return None
    return np.linalg.lstsq(np.column_stack([np.ones(ok.sum()), X[ok]]), y[ok], rcond=None)[0][1:]
def impulse_t6(a, phi, kappa, r, eps, eta, q, L=120):
    x = np.zeros(L); dd = np.zeros(L); f = np.zeros(L)
    for t in range(L):
        if t > 0:
            f[t] = (1 - eta) * f[t - 1] + eta * q * x[t - 1]
        tot = (1.0 if t == 0 else 0.0) + f[t]
        x[t] = (a * x[t - 1] + phi * dd[t - 1] if t else 0) + kappa * tot
        dd[t] = ((1 - r) * dd[t - 1] if t else 0) + eps * tot
        if abs(x[t]) > 1e9:
            return np.full(L, np.nan)
    return x
def fit_t6(p, u, d, x):
    dp = np.diff(p, prepend=np.nan)
    bp = ols(dp[1:], np.column_stack([u[1:], d[:-1], x[:-1]]))
    bd = ols(d[1:], np.column_stack([d[:-1], u[1:]]))
    bm = ols(u[1:], np.column_stack([u[:-1], x[:-1]]))
    if bp is None or bd is None or bm is None:
        return None
    kappa, phi, a = bp[0], bp[1], 1 + bp[2]
    r, eps, eta = 1 - bd[0], bd[1], 1 - bm[0]
    q = bm[1] / eta if abs(eta) > 1e-6 else np.nan
    h = impulse_t6(a, phi, kappa, r, eps, eta, q)
    ok = np.all(np.isfinite(h)) and abs(h[0]) > 1e-12
    return {"kappa": kappa, "phi": phi, "a": a, "r": r, "eps": eps, "eta": eta, "q": q,
            "C2": phi * eps - (1 - a) * kappa, "C3": phi * eps + kappa ** 2 * eta * q - (1 - a) * kappa,
            "peak_ratio": float(np.max(h) / h[0]) if ok else np.nan, "peak_minute": int(np.argmax(h)) if ok else -1}
def loop_gain(p, u, span=30):
    x = p - pd.Series(p).ewm(span=span, adjust=False).mean().to_numpy()
    f = np.roll(u, -1)[:-1]; x = x[:-1]
    Z = np.column_stack([x, f])
    Y, X = Z[1:], np.column_stack([np.ones(len(Z) - 1), Z[:-1]])
    ok = np.all(np.isfinite(Y), 1) & np.all(np.isfinite(X), 1)
    if ok.sum() < 100:
        return np.nan
    J = np.linalg.lstsq(X[ok], Y[ok], rcond=None)[0][1:].T
    return float(J[0, 1] * J[1, 0] / (1 - J[1, 1])) if abs(1 - J[1, 1]) > 1e-9 else np.nan
def auc_(y, s):
    pos, neg = s[y == 1], s[y == 0]
    if len(pos) == 0 or len(neg) == 0:
        return np.nan
    rk = stats.rankdata(np.concatenate([pos, neg]))
    return (rk[:len(pos)].sum() - len(pos) * (len(pos) + 1) / 2) / (len(pos) * len(neg))
def cw_test(y, fb, ff, lags=6):
    adj = (y - fb) ** 2 - ((y - ff) ** 2 - (fb - ff) ** 2)
    m = adj.mean(); e = adj - m; n = len(adj)
    lrv = e @ e / n + 2 * sum((1 - l / (lags + 1)) * (e[l:] @ e[:-l]) / n for l in range(1, lags + 1))
    t_ = m / np.sqrt(lrv / n)
    return float(t_), float(1 - stats.norm.cdf(t_))
def design(D, cols, syms):
    X = [np.ones(len(D))] + [D[c].to_numpy() for c in cols] + [(D["symbol"] == s).to_numpy(float) for s in syms[1:]]
    return np.column_stack(X)
def evaluate(sub, outcome, cand):
    A0 = sub[sub["split"] == "train"].copy(); B0 = sub[sub["split"] == "test"].copy()
    lo, hi = A0[cand].quantile([0.01, 0.99]); A0["cand_w"] = A0[cand].clip(lo, hi); B0["cand_w"] = B0[cand].clip(lo, hi)
    need = CTRL + ["cand_w", outcome]
    A = A0.dropna(subset=need); B = B0.dropna(subset=need).sort_values(["date", "symbol", "hour"])
    if len(A) < 200 or len(B) < 200 or A[outcome].std() < 1e-8 or A["cand_w"].std() < 1e-8:
        return {"skipped": "too few blocks or no variation in outcome/candidate", "n_train": len(A), "n_test": len(B),
                "clark_west_p": np.nan, "delta_r2": np.nan, "delta_r2_ci": [np.nan, np.nan]}
    B = B[B["symbol"].isin(set(A["symbol"]))]
    syms = sorted(A["symbol"].unique()); y = B[outcome].to_numpy()
    bb = np.linalg.lstsq(design(A, CTRL, syms), A[outcome].to_numpy(), rcond=None)[0]
    bf = np.linalg.lstsq(design(A, CTRL + ["cand_w"], syms), A[outcome].to_numpy(), rcond=None)[0]
    fb = design(B, CTRL, syms) @ bb; ff = design(B, CTRL + ["cand_w"], syms) @ bf
    ybar = A[outcome].mean(); sst = np.sum((y - ybar) ** 2)
    t_, p_ = cw_test(y, fb, ff)
    tail = (y > A[outcome].quantile(0.9)).astype(int)
    Rb = rng(88); dd = B["date"].to_numpy(); ud = np.unique(dd)
    pos = {d: np.where(dd == d)[0] for d in ud}; dR2, dA = [], []
    for _ in range(2000):
        idx = np.concatenate([pos[d] for d in Rb.choice(ud, len(ud))])
        s_ = np.sum((y[idx] - ybar) ** 2)
        dR2.append((np.sum((y[idx] - fb[idx]) ** 2) - np.sum((y[idx] - ff[idx]) ** 2)) / s_)
        dA.append(auc_(tail[idx], ff[idx]) - auc_(tail[idx], fb[idx]))
    return {"n_train": len(A), "n_test": len(B), "n_symbols": len(syms),
            "r2_base": 1 - np.sum((y - fb) ** 2) / sst, "r2_full": 1 - np.sum((y - ff) ** 2) / sst,
            "delta_r2": (np.sum((y - fb) ** 2) - np.sum((y - ff) ** 2)) / sst,
            "delta_r2_ci": [float(np.nanquantile(dR2, .025)), float(np.nanquantile(dR2, .975))],
            "auc_base": auc_(tail, fb), "auc_full": auc_(tail, ff),
            "delta_auc_ci": [float(np.nanquantile(dA, .025)), float(np.nanquantile(dA, .975))],
            "clark_west_t": t_, "clark_west_p": p_, "coef_candidate_train": float(bf[len(CTRL) + 1])}

# ---------------------------------------------------------------------------------------------
# Part 6 additions
# ---------------------------------------------------------------------------------------------
H_GUARD = 60          # guard horizon (minutes) for the summed and peak impulse response
CTRL = ["rv_60", "rv_360", "abs_ret_60", "flow_imb_60", "log_vol_60", "log_depth1", "imbalance", "oi_chg_60"]
IMPACT = ["kappa_hat", "kappa_sq", "kyle_lam", "amihud_60"]
TAGGED = ["n_top_pos", "d_top_pos_60", "n_all", "d_all_60"]

def _last_valid(v, lo, hi):
    w = v[lo:hi]; w = w[np.isfinite(w)]
    return w[-1] if len(w) else np.nan

def day_blocks6(s, d, flow_scale):
    """One row per hour 06-23 UTC. Features use minutes [t-360, t); outcomes use (t, t+60]."""
    df = PANEL[(s, d)]
    p, u, dd_, x, tot = prep(df, flow_scale)
    oi = df["oi"].ffill().to_numpy(); vol = df["vol"].to_numpy(); dp = np.diff(p, prepend=np.nan)
    ntag = {c: ((df[col] - 1) / (df[col] + 1)).to_numpy() for c, col in [("top_pos", "top_pos_ratio"), ("all", "acc_ratio")]}
    rows = []
    for hr in range(6, 24):
        t = hr * 60
        w, h, f = slice(t - 360, t), slice(t - 60, t), slice(t, min(t + 61, 1440))
        est = fit_t6(p[w], u[w], dd_[w], x[w])
        row = {"symbol": s, "date": d, "hour": hr}
        if est is not None:
            t1, t2, t3 = est["phi"] * est["eps"], est["kappa"] ** 2 * est["eta"] * est["q"], (1 - est["a"]) * est["kappa"]
            imp = impulse_t6(est["a"], est["phi"], est["kappa"], est["r"], est["eps"], est["eta"], est["q"], L=H_GUARD)
            ok = np.all(np.isfinite(imp)) and abs(imp[0]) > 1e-12
            S60 = float(np.sum(np.abs(imp))) if ok else np.nan
            P60 = float(np.max(np.abs(imp))) if ok else np.nan
            row.update({"C3_hat": est["C3"], "term_depth": t1, "term_momentum": t2, "term_reversion": t3,
                        "kappa_hat": est["kappa"], "a_hat": est["a"], "S60": S60, "P60": P60,
                        "gain_multiple": S60 / abs(imp[0]) if ok else np.nan,
                        "log_halflife": float(np.log(1.0 / (1 - est["a"]))) if 0 < 1 - est["a"] < 1 else np.nan})
        # depth-free version of C3 (prices and flow only), for venues without depth data (Part 7)
        bp0 = ols(dp[w][1:], np.column_stack([u[w][1:], x[w][:-1]]))
        bm0 = ols(u[w][1:], np.column_stack([u[w][:-1], x[w][:-1]]))
        if bp0 is not None and bm0 is not None and abs(1 - bm0[0]) > 1e-6:
            k0, a0, e0 = bp0[0], 1 + bp0[1], 1 - bm0[0]
            row["C3_flow"] = k0 ** 2 * e0 * (bm0[1] / e0) - (1 - a0) * k0
        uu, yy = u[w][1:], dp[w][1:]; okk = np.isfinite(uu) & np.isfinite(yy)
        kl = float(np.polyfit(uu[okk], yy[okk], 1)[0]) if okk.sum() > 60 and np.std(uu[okk]) > 0 else np.nan
        v60 = np.nansum(vol[h]); au = np.abs(u[w]); au = au[np.isfinite(au)]
        row.update({
            "rv_60": np.nanstd(np.diff(p[h])), "rv_360": np.nanstd(np.diff(p[w])), "abs_ret_60": abs(p[t - 1] - p[t - 60]),
            "flow_imb_60": np.nansum(u[h]) / (np.nansum(np.abs(u[h])) + 1e-12), "log_vol_60": np.log1p(v60),
            "log_depth1": np.log(tot[t - 1]) if np.isfinite(tot[t - 1]) and tot[t - 1] > 0 else np.nan,
            "imbalance": dd_[t - 1],
            "oi_chg_60": (oi[t - 1] / oi[t - 60] - 1) * 100 if np.isfinite(oi[t - 60]) and oi[t - 60] > 0 else np.nan,
            "kyle_lam": kl,
            "amihud_60": np.log(abs(p[t - 1] - p[t - 60]) / v60 + 1e-9) if v60 > 0 and np.isfinite(p[t - 60]) else np.nan,
            "U6": float(np.quantile(au, 0.95)) if len(au) > 60 else np.nan,
            "n_top_pos": _last_valid(ntag["top_pos"], t - 60, t),
            "d_top_pos_60": _last_valid(ntag["top_pos"], t - 60, t) - _last_valid(ntag["top_pos"], t - 120, t - 60),
            "n_all": _last_valid(ntag["all"], t - 60, t),
            "d_all_60": _last_valid(ntag["all"], t - 60, t) - _last_valid(ntag["all"], t - 120, t - 60),
            "MAE_60": float(np.nanmax(np.abs(p[f] - p[t - 1]))),
            "umax_next": float(np.nanmax(np.abs(u[f])))})
        rows.append(row)
    return rows

def winsor_train(D, cols):
    D = D.copy()
    for c in cols:
        lo, hi = D.loc[D["split"] == "train", c].quantile([0.01, 0.99])
        D[c] = D[c].clip(lo, hi)
    return D

def evaluate_g(sub, outcome, cands, ctrl, B=2000, level=0.95, seed=88):
    """Out-of-sample evaluation, as in Parts 4-5, for one or more candidates on top of `ctrl`.
    Candidates are winsorized at training 1%/99%; symbol fixed effects; date-block bootstrap."""
    cands = [cands] if isinstance(cands, str) else list(cands)
    A0 = sub[sub["split"] == "train"].copy(); B0 = sub[sub["split"] == "test"].copy()
    cw = []
    for c in cands:
        lo, hi = A0[c].quantile([0.01, 0.99]); A0[c + "_w"] = A0[c].clip(lo, hi); B0[c + "_w"] = B0[c].clip(lo, hi); cw.append(c + "_w")
    need = ctrl + cw + [outcome]
    A = A0.dropna(subset=need); Bt = B0.dropna(subset=need)
    Bt = Bt[Bt["symbol"].isin(set(A["symbol"]))].sort_values(["date", "symbol", "hour"])
    if len(A) < 200 or len(Bt) < 200 or A[outcome].std() < 1e-8 or any(A[c].std() < 1e-12 for c in cw):
        return {"skipped": "too few blocks or no variation", "n_train": len(A), "n_test": len(Bt),
                "clark_west_p": np.nan, "delta_r2": np.nan, "delta_r2_ci": [np.nan, np.nan]}
    syms = sorted(A["symbol"].unique()); y = Bt[outcome].to_numpy()
    bb = np.linalg.lstsq(design(A, ctrl, syms), A[outcome].to_numpy(), rcond=None)[0]
    bf = np.linalg.lstsq(design(A, ctrl + cw, syms), A[outcome].to_numpy(), rcond=None)[0]
    fb = design(Bt, ctrl, syms) @ bb; ff = design(Bt, ctrl + cw, syms) @ bf
    ybar = A[outcome].mean(); sst = np.sum((y - ybar) ** 2)
    t_, p_ = cw_test(y, fb, ff)
    tail = (y > A[outcome].quantile(0.9)).astype(int)
    Rb = rng(seed); dd = Bt["date"].to_numpy(); ud = np.unique(dd)
    pos = {d: np.where(dd == d)[0] for d in ud}; dR2, dA = [], []
    for _ in range(B):
        idx = np.concatenate([pos[d] for d in Rb.choice(ud, len(ud))])
        s_ = np.sum((y[idx] - ybar) ** 2)
        dR2.append((np.sum((y[idx] - fb[idx]) ** 2) - np.sum((y[idx] - ff[idx]) ** 2)) / s_)
        dA.append(auc_(tail[idx], ff[idx]) - auc_(tail[idx], fb[idx]))
    qa = (1 - level) / 2
    return {"n_train": len(A), "n_test": len(Bt), "n_symbols": len(syms), "n_test_dates": len(ud),
            "r2_base": 1 - np.sum((y - fb) ** 2) / sst, "r2_full": 1 - np.sum((y - ff) ** 2) / sst,
            "delta_r2": (np.sum((y - fb) ** 2) - np.sum((y - ff) ** 2)) / sst,
            "delta_r2_ci": [float(np.nanquantile(dR2, qa)), float(np.nanquantile(dR2, 1 - qa))],
            "auc_base": auc_(tail, fb), "auc_full": auc_(tail, ff),
            "delta_auc_ci": [float(np.nanquantile(dA, qa)), float(np.nanquantile(dA, 1 - qa))],
            "clark_west_t": t_, "clark_west_p": p_, "ci_level": level,
            "coef_candidates_train": {c: float(bf[len(ctrl) + 1 + i]) for i, c in enumerate(cands)}}


# ---------------------------------------------------------------------------------------------
# Part 9 additions: volatility-forecasting benchmarks (all from data before t, same day)
# ---------------------------------------------------------------------------------------------
HOURS = list(range(7, 24))                                  # hour 6 is the omitted category
VOLBENCH = ["rv_15", "rv_day", "ewma_vol", "mae_lag60"] + [f"hr_{h}" for h in HOURS]
def day_blocks9(s, d, flow_scale):
    rows = day_blocks6(s, d, flow_scale)
    df = PANEL[(s, d)]; p = np.log(df["close"].to_numpy()) * 1e4; dp = np.diff(p, prepend=np.nan)
    ew = pd.Series(np.nan_to_num(dp) ** 2).ewm(alpha=0.06, adjust=False).mean().to_numpy()   # RiskMetrics lambda = 0.94
    for row in rows:
        t = row["hour"] * 60
        row["rv_15"] = np.nanstd(dp[t - 15:t]); row["rv_day"] = np.nanstd(dp[1:t])
        row["ewma_vol"] = float(np.sqrt(ew[t - 1]))
        row["mae_lag60"] = float(np.nanmax(np.abs(p[t - 60:t] - p[t - 61])))
        for h in HOURS:
            row[f"hr_{h}"] = float(row["hour"] == h)
    return rows


# ---------------------------------------------------------------------------------------------
# Part 10: Bybit order-book files -> 1-minute depth; Bybit trades -> 1-minute price, flow, volume
# ---------------------------------------------------------------------------------------------
OB_NAME = re.compile(r"(\d{4}-\d{2}-\d{2})_([A-Z0-9]+)USDT_ob(\d+)\.data$")

DEPTH_LEVELS = 200          # the pre-registered book is the top 200 levels; 2024 files carry 500, so only the best 200 are used
def book_minutes(fh, date, band=0.01, levels=DEPTH_LEVELS):
    """Rebuild the book from one day's messages (a 'snapshot' resets it; a 'delta' sets levels, size 0 deletes).
    Returns the book at the END of each UTC minute: best bid/ask and $ notional within 1% of mid ($M),
    counting only the best `levels` price levels on each side (identical for ob200 and ob500 files)."""
    t0 = int(pd.Timestamp(date, tz="UTC").timestamp() * 1000)
    bids, asks = {}, {}
    out = np.full((1440, 4), np.nan); cur = None; last_u = None; gaps = 0; n_msg = 0; n_snap = 0; crossed = 0
    def rec(m):
        nonlocal crossed
        if not (0 <= m < 1440) or not bids or not asks:
            return
        bb, ba = max(bids), min(asks)
        if bb >= ba:
            crossed += 1; return
        mid = 0.5 * (bb + ba)
        bl = sorted(bids, reverse=True)[:levels]; al = sorted(asks)[:levels]
        out[m] = (bb, ba, sum(p * bids[p] for p in bl if p >= mid * (1 - band)) / 1e6,
                  sum(p * asks[p] for p in al if p <= mid * (1 + band)) / 1e6)
    for line in fh:
        if not line.strip():
            continue
        r = json.loads(line); n_msg += 1
        m = (r["ts"] - t0) // 60000
        if cur is not None and m != cur:
            rec(cur)
        cur = m; d = r["data"]
        if r["type"] == "snapshot":
            n_snap += 1
            bids = {float(p): float(q) for p, q in d["b"]}; asks = {float(p): float(q) for p, q in d["a"]}
        else:
            if last_u is not None and d["u"] != last_u + 1:
                gaps += 1
            for side, book in (("b", bids), ("a", asks)):
                for p, q in d[side]:
                    p = float(p); q = float(q)
                    if q == 0:
                        book.pop(p, None)
                    else:
                        book[p] = q
        last_u = d.get("u", last_u)
    if cur is not None:
        rec(cur)
    df = pd.DataFrame(out, columns=["best_bid", "best_ask", "bid1", "ask1"])
    st = {"messages": n_msg, "snapshots": n_snap, "u_gaps": gaps, "crossed_minutes": crossed,
          "depth_rule": f"top{levels}_within_{band:g}",
          "book_minute_coverage": float(df["bid1"].notna().mean())}
    return df, st

def _convert_member(name, raw, source):
    mt = OB_NAME.search(os.path.basename(name))
    if not mt:
        return None
    date, coin = mt.group(1), mt.group(2)
    df, st = book_minutes(io.StringIO(raw.decode()), date)
    st.update({"coin": coin, "date": date, "depth_levels": int(mt.group(3)), "source_file": source,
               "sha256": hashlib.sha256(raw).hexdigest(), "bytes": len(raw)})
    d_ = os.path.join(OBMIN, coin); os.makedirs(d_, exist_ok=True)
    df.to_parquet(os.path.join(d_, f"{date}.parquet"))
    json.dump(_clean(st), open(os.path.join(d_, f"{date}.json"), "w"))
    return (coin, date)

def convert_zip(path):
    """Turn one uploaded Bybit file into small per-day book summaries; returns the (coin, date) pairs written.
    Accepts a daily .zip, a bundle .zip holding daily .zip files (one Bybit date-range download), or a loose .data file."""
    done = []
    if path.endswith(".data"):
        got = _convert_member(path, open(path, "rb").read(), os.path.basename(path))
        return [got] if got else []
    def walk(zf, src):
        for name in zf.namelist():
            if name.endswith("/"):
                continue
            if name.lower().endswith(".zip"):
                with zipfile.ZipFile(io.BytesIO(zf.read(name))) as inner:
                    walk(inner, src)
            else:
                got = _convert_member(name, zf.read(name), src)
                if got:
                    done.append(got)
    with zipfile.ZipFile(path) as z:
        walk(z, os.path.basename(path))
    return done

def have_book(coin, date):
    js = os.path.join(OBMIN, coin, f"{date}.json")
    return os.path.exists(js) and json.load(open(js)).get("depth_rule", "").startswith(f"top{DEPTH_LEVELS}_")

def trade_minutes(coin, date):
    """Minute bars from Bybit trades (same code as Part 7): last price, signed taker notional and volume ($M)."""
    sym = f"{coin}USDT"
    d_ = os.path.join(TRMIN, coin); os.makedirs(d_, exist_ok=True)
    pq, js = os.path.join(d_, f"{date}.parquet"), os.path.join(d_, f"{date}.json")
    if os.path.exists(js):
        st = json.load(open(js)); return (pd.read_parquet(pq) if os.path.exists(pq) else None), st
    code, raw = _http(bb_url(sym, date), timeout=180)
    if code != 200 or not raw:
        st = {"coin": coin, "date": date, "reason": f"http {code}", "day_ok": False}
        json.dump(st, open(js, "w")); return None, st
    DL_LOG.append({"symbol": sym, "date": date, "sha256": hashlib.sha256(raw).hexdigest(), "bytes": len(raw)})
    tr = pd.read_csv(io.BytesIO(gzip.decompress(raw)))
    tr.columns = [c.strip().lower() for c in tr.columns]
    ts = pd.to_numeric(tr["timestamp"], errors="coerce").to_numpy()
    ts = np.where(ts > 1e11, ts / 1000.0, ts)
    m = ((ts - pd.Timestamp(date, tz="UTC").timestamp()) // 60).astype("int64")
    px = pd.to_numeric(tr["price"], errors="coerce").to_numpy(); sz = pd.to_numeric(tr["size"], errors="coerce").to_numpy()
    notional = px * sz / 1e6; sign = np.where(tr["side"].astype(str).str.lower().str.startswith("b"), 1.0, -1.0)
    ok = (m >= 0) & (m < 1440) & np.isfinite(px) & np.isfinite(notional)
    T = pd.DataFrame({"m": m[ok], "ts": ts[ok], "px": px[ok], "flow": (sign * notional)[ok], "vol": notional[ok]}).sort_values("ts")
    idx = np.arange(1440); g = T.groupby("m")
    out = pd.DataFrame(index=idx)
    out["close"] = g["px"].last().reindex(idx).ffill().bfill()
    out["flow"] = g["flow"].sum().reindex(idx).fillna(0.0)
    out["vol"] = g["vol"].sum().reindex(idx).fillna(0.0)
    cover = float(g.size().reindex(idx).notna().mean())
    st = {"coin": coin, "date": date, "n_trades": int(ok.sum()), "trade_minute_coverage": cover,
          "day_ok": bool(cover >= 0.80 and out["close"].notna().all())}
    out.to_parquet(pq); json.dump(_clean(st), open(js, "w"))
    return out, st

def build_day_bybit_ob(coin, date):
    """Merge: price, flow and volume from trades (as Binance klines in Parts 4-9); depth within 1% from the book."""
    tr, st_t = trade_minutes(coin, date)
    st_b = json.load(open(os.path.join(OBMIN, coin, f"{date}.json")))
    st = {"symbol": coin, "date": date, **{k: v for k, v in st_t.items() if k not in ("coin", "date")},
          **{k: v for k, v in st_b.items() if k not in ("coin", "date")}}
    if tr is None or not st_t.get("day_ok"):
        st["depth_day_ok"] = False; return None, st
    bk = pd.read_parquet(os.path.join(OBMIN, coin, f"{date}.parquet"))
    out = tr.copy()
    mid = 0.5 * (bk["best_bid"] + bk["best_ask"])
    far = (np.abs(mid / out["close"] - 1) > 0.03).to_numpy()      # book far from traded price: treat as missing
    for c in ["bid1", "ask1"]:
        v = bk[c].to_numpy().copy(); v[far] = np.nan
        out[c] = pd.Series(v).ffill(limit=2).to_numpy()
    for c in ["oi", "top_pos_ratio", "top_acc_ratio", "acc_ratio", "taker_ratio"]:
        out[c] = np.nan
    cov = float(pd.Series(out["bid1"]).notna().mean())
    st.update({"merged_depth_coverage": cov, "book_far_from_trades": int(far.sum()),
               "depth_day_ok": bool(cov >= 0.80 and st_b.get("u_gaps", 99) <= 5 and st_b.get("snapshots", 0) >= 1)})
    out["symbol"], out["date"] = coin, date
    return out, st

## Pre-registration (before uploading any test-day file)

In [ ]:
# ============================================================================
# CELL V2 — PRE-REGISTRATION (run ONCE, BEFORE you upload any test-day file; V3 refuses to run without it)
# It downloads P10_bybit_ob.json: e-mail it to yourself before uploading data. Never edit this cell afterwards.
# ============================================================================
CTRL10 = ["rv_60", "rv_360", "abs_ret_60", "flow_imb_60", "log_vol_60", "log_depth1", "imbalance"]   # Part 9 minus open interest
BASE10 = CTRL10 + IMPACT + VOLBENCH
PREREG_P10 = {
    "question": "A second venue with real depth: in the 20 thin-set coins on Bybit USDT perpetuals, does C3_hat forecast "
                "next-hour MAE beyond the Part 9 baseline (volatility forecasting, price impact, depth, hour effects)?",
    "relation_to_part7": "Part 7 tested Bybit with trades only (depth-free C3_flow) and failed. This is a new test with "
                         "order-book depth on days Part 7 never used. Both results are reported.",
    "data": "Bybit history-data, Contract, OB Data (daily ob200 files: one snapshot plus every update of the top 200 levels), "
            "rebuilt to the book at the end of each UTC minute; Bybit public trade archive for price, signed taker flow and "
            "volume. Price = last trade (as Binance klines in Parts 4-9). Depth = $ notional within 1% of mid.",
    "day_validity": "trade minutes >= 80%; merged depth minutes >= 80%; <= 5 update-id gaps; a book whose mid is > 3% from the "
                    "last trade is treated as missing",
    "coins": COINS, "train": TRAIN_D, "test": TEST_D,
    "inclusion": "a coin enters if it has >= 8 valid training days and >= 24 valid test days; coins Bybit does not list are excluded",
    "unit": "symbol-hour blocks, hours 6-23 UTC; features from minutes before t on the same day; outcome MAE over (t, t+60]",
    "baseline": BASE10 + ["symbol fixed effects"],
    "primary": {"sample": "all included coins", "outcome": "MAE_60", "candidate": "C3_hat (winsorized at train 1%/99%)",
                "success": "Clark-West one-sided p < 0.05 AND 95% date-block bootstrap CI of dR2 above 0"},
    "secondary": ["MAE | S60", "MAE | gain_multiple", "MAE | log_halflife", "MAE | term_reversion", "MAE | term_momentum",
                  "MAE | C3_flow"],
    "inference": "Clark-West (Newey-West 6 lags); 2,000 date-block bootstrap draws (all symbols on a date together); secondary Holm-adjusted",
    "runs": "V6 is run once; every run is counted in prereg/test_set_unlocks_p10.json",
    "previous_hashes": {"part6": "65acda36186147bd6e959ce8c56b9958b7a7f94a01f8784bc82f2b1512360215",
                        "part9": "71983f1a9d2ca23cafe5ce4de5d5d9ddb406529ceaf5aec57a7dee81a75f846d"},
}
spec_txt10 = json.dumps(PREREG_P10, indent=2, sort_keys=True)
P10_HASH = hashlib.sha256(spec_txt10.encode()).hexdigest()
p10_path = os.path.join(PREREG, "P10_bybit_ob.json")
if not os.path.exists(p10_path):
    open(p10_path, "w").write(spec_txt10); print("pre-registration written:", p10_path)
    if IN_COLAB:
        from google.colab import files
        files.download(p10_path)
elif hashlib.sha256(open(p10_path).read().encode()).hexdigest() != P10_HASH:
    raise RuntimeError("The spec differs from the registered file. Do not edit the spec after registering it.")
print("spec SHA-256:", P10_HASH)

## Automatic download (run this instead of uploading by hand)

In [ ]:
# ============================================================================
# CELL V3A — AUTOMATIC DOWNLOAD (replaces manual downloading; safe to rerun, finished days are skipped)
# Fetches every coin-day of the pre-registered design straight from Bybit's file server, converts it, deletes the zip.
# Takes about 1-2 hours. If Colab disconnects: run V0, V1, V2, then this cell again; it continues where it stopped.
# ============================================================================
assert os.path.exists(p10_path) and hashlib.sha256(open(p10_path).read().encode()).hexdigest() == P10_HASH, "run V2 first"
OBURL = "https://quote-saver.bycsi.com/orderbook/linear/{s}/{d}_{s}_ob{n}.data.zip"
AUTO = "/content/bybit_ob_auto" if IN_COLAB else os.path.abspath("bybit_ob_auto")
os.makedirs(AUTO, exist_ok=True)
OB_LOG = []
def fetch_ob(coin, date):
    if have_book(coin, date):
        return "already converted"
    s = f"{coin}USDT"
    for n in ((500, 200) if date < "2025" else (200, 500)):
        url = OBURL.format(s=s, d=date, n=n)
        try:
            code, raw = _http(url, timeout=600)
        except Exception as e:
            return f"network error: {e!s:.120}"
        if code == 200 and raw:
            path = os.path.join(AUTO, f"{date}_{s}_ob{n}.data.zip")
            open(path, "wb").write(raw)
            OB_LOG.append({"coin": coin, "date": date, "url": url, "zip_sha256": hashlib.sha256(raw).hexdigest(), "bytes": len(raw)})
            try:
                got = convert_zip(path)
            finally:
                os.remove(path)
            return f"ok ob{n}" if got else "zip had no order-book file"
    return "not on Bybit"
# 1) probe with the link you copied from Chrome
code, _ = _http(OBURL.format(s="ALICEUSDT", d="2024-03-12", n=500), timeout=300)
print("Bybit order-book server reachable from Colab:", code == 200, f"(HTTP {code})")
assert code == 200, "Colab cannot reach quote-saver.bycsi.com; use manual downloads + V3, and tell Claude."
# 2) everything else
jobs = [(c, d) for c in COINS for d in TRAIN_D + TEST_D]
res = {}
t_start = time.time()
with ThreadPoolExecutor(4) as ex:
    futs = {ex.submit(fetch_ob, c, d): (c, d) for c, d in jobs}
    for i, f in enumerate(as_completed(futs), 1):
        res[futs[f]] = f.result()
        if i % 20 == 0:
            el = (time.time() - t_start) / 60
            print(f"   {i}/{len(jobs)} coin-days, {el:.0f} min elapsed, about {el / i * (len(jobs) - i):.0f} min left")
summ = pd.Series(res).value_counts()
print("\n", summ.to_string())
for bname, coins in BATCHES.items():
    for c in coins:
        miss = [d for d in TRAIN_D + TEST_D if not have_book(c, d)]
        if miss:
            print(f"   {c}: {len(miss)} days not available -> {sorted({d[:7] for d in miss})}")
pd.DataFrame(OB_LOG).to_csv(os.path.join(RESULTS, "p10_orderbook_manifest_sha256.csv"),
                            mode="a", header=not os.path.exists(os.path.join(RESULTS, "p10_orderbook_manifest_sha256.csv")), index=False)
record("V3A_orderbook_download", None, "audit", {"status_counts": summ.to_dict(),
       "missing": {c: [d for d in TRAIN_D + TEST_D if not have_book(c, d)] for c in COINS}})
print("\nDone. Next: run V4 (you can skip V3).")

## Manual fallback only: upload a batch, then run V3 (skip if V3A worked)

In [ ]:
# ============================================================================
# CELL V3 — CONVERT UPLOADED ZIPS (run after EVERY batch you upload; safe to rerun)
# Put Bybit .zip files in /content/bybit_ob (Colab Files panel) or in Drive MyDrive/ceiling_suite/bybit_ob.
# Each zip becomes a small summary saved in Drive, then the zip is deleted to free space.
# ============================================================================
assert os.path.exists(p10_path) and hashlib.sha256(open(p10_path).read().encode()).hexdigest() == P10_HASH, "run V2 first"
zips = sorted(set(sum([glob.glob(os.path.join(u, "**", "*.zip"), recursive=True) +
                      glob.glob(os.path.join(u, "**", "*.data"), recursive=True) for u in UPLOAD_DIRS], [])))
print(f"{len(zips)} files found")
written, bad = [], []
for i, zp in enumerate(zips, 1):
    try:
        got = convert_zip(zp); written += got
        if got:
            os.remove(zp)
        else:
            bad.append((os.path.basename(zp), "no ob .data file inside"))
    except Exception as e:
        bad.append((os.path.basename(zp), repr(e)[:200]))
    if i % 10 == 0:
        print(f"   {i}/{len(zips)}")
print(f"converted {len(written)} coin-days; problems: {len(bad)}")
for b in bad:
    print("   PROBLEM:", b)
# What is still missing, batch by batch (this is your download checklist)
WANT = TRAIN_D + TEST_D
for bname, coins in BATCHES.items():
    miss = {c: [d for d in WANT if not have_book(c, d)] for c in coins}
    n = sum(len(v) for v in miss.values())
    print(f"\nBatch {bname} {coins}: {len(coins) * len(WANT) - n}/{len(coins) * len(WANT)} coin-days present")
    for c, v in miss.items():
        if v:
            wins = sorted({d[:7] for d in v})
            print(f"   {c}: missing {len(v)} days, in windows {wins}")
off = [(c, d) for c in COINS for d in WANT if have_book(c, d) and
       json.load(open(os.path.join(OBMIN, c, f"{d}.json"))).get("u_gaps", 0) > 5]
if off:
    print("\nDays with broken update sequences (will be dropped):", off)
if IN_COLAB:
    print("\nIf you used the Drive folder: empty the Google Drive trash now to get the space back.")

## Trades, merge, hourly blocks

In [ ]:
# ============================================================================
# CELL V4 — TRADES + MERGE (downloads Bybit trades for every converted day; cached in Drive)
# ============================================================================
assert os.path.exists(p10_path) and hashlib.sha256(open(p10_path).read().encode()).hexdigest() == P10_HASH, "run V2 first"
jobs = [(c, d) for c in COINS for d in TRAIN_D + TEST_D if have_book(c, d)]
print(f"{len(jobs)} coin-days with a book file")
PANEL, VST = {}, []
with ThreadPoolExecutor(THREADS) as ex:
    futs = {ex.submit(build_day_bybit_ob, c, d): (c, d) for c, d in jobs}
    for i, f in enumerate(as_completed(futs), 1):
        c, d = futs[f]
        try:
            df, st = f.result(); VST.append(st)
            if df is not None and st.get("depth_day_ok"):
                PANEL[(c, d)] = df
        except Exception as e:
            VST.append({"symbol": c, "date": d, "reason": f"error: {e}"})
        if i % 100 == 0:
            print(f"   {i}/{len(jobs)}")
vst = pd.DataFrame(VST)
ntr = {c: sum(1 for (s, d) in PANEL if s == c and d <= TRAIN_END) for c in COINS}
nte = {c: sum(1 for (s, d) in PANEL if s == c and d > TRAIN_END) for c in COINS}
SYMBOLS = sorted(c for c in COINS if ntr[c] >= 8 and nte[c] >= 24)
GROUP = {s: "thin" for s in SYMBOLS}
OK = sorted((s, d) for (s, d) in PANEL if s in SYMBOLS)
FLOW_SCALE = {}
for s in SYMBOLS:
    v = [np.nanmedian(np.abs(PANEL[(s, d)]["flow"].to_numpy())) for d in TRAIN_D if (s, d) in PANEL]
    v = [x for x in v if np.isfinite(x) and x > 0]
    FLOW_SCALE[s] = float(np.median(v)) if v else 1.0
print("valid days (train / test):", {c: (ntr[c], nte[c]) for c in COINS})
print(f"included coins: {len(SYMBOLS)}", SYMBOLS)
med_depth = {s: float(np.nanmedian([np.nanmedian((PANEL[(s, d)]["bid1"] + PANEL[(s, d)]["ask1"]).to_numpy())
                                     for d in TRAIN_D if (s, d) in PANEL])) for s in SYMBOLS}
print("median depth within 1% on training days ($M):", {k: round(v, 3) for k, v in med_depth.items()})
vst.to_csv(os.path.join(RESULTS, "p10_day_status.csv"), index=False)
pd.DataFrame(DL_LOG).to_csv(os.path.join(RESULTS, "p10_trades_manifest_sha256.csv"), index=False)
record("V4_build", None, "audit", {"requested": len(jobs), "valid": len(PANEL), "train_days": ntr, "test_days": nte,
       "included": SYMBOLS, "flow_scale": FLOW_SCALE, "median_depth1_musd": med_depth})

In [ ]:
# ============================================================================
# CELL V5 — HOURLY BLOCKS (no outcome is looked at here)
# ============================================================================
assert SYMBOLS, "no coin passed the inclusion rule; tell Claude"
rows = []
for i, (s, d) in enumerate(OK):
    rows += day_blocks9(s, d, FLOW_SCALE[s])
    if (i + 1) % 100 == 0:
        print(f"   blocks: {i+1}/{len(OK)}")
TB = pd.DataFrame(rows)
TB["split"] = np.where(TB["date"] <= TRAIN_END, "train", "test")
TB["kappa_sq"] = TB["kappa_hat"] ** 2
TB = winsor_train(TB, IMPACT + ["rv_15", "rv_day", "ewma_vol", "mae_lag60"])
TB.to_parquet(os.path.join(BASE, "data_bybit_ob", "blocks_p10.parquet"))
print(TB.groupby("split").size())
record("V5_blocks", None, "audit", {"blocks": TB.groupby("split").size().to_dict(),
       "coverage": {c: float(TB[c].notna().mean()) for c in ["C3_hat", "S60", "log_depth1", "imbalance", "ewma_vol"]}})

## The test: run ONCE

In [ ]:
# ============================================================================
# CELL V6 — THE TEST (RUN ONCE; every run is counted). It zips the results and downloads p10_results.zip.
# ============================================================================
assert hashlib.sha256(open(p10_path).read().encode()).hexdigest() == P10_HASH, "spec changed after registration"
pth10 = os.path.join(PREREG, "test_set_unlocks_p10.json")
n_unl10 = (json.load(open(pth10))["n"] if os.path.exists(pth10) else 0) + 1
json.dump({"n": n_unl10, "utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()), "spec_sha256": P10_HASH}, open(pth10, "w"))
def holm(pd_):
    items = sorted((v if np.isfinite(v) else 1.0, k) for k, v in pd_.items()); m = len(items); out, run = {}, 0.0
    for j, (p_, k) in enumerate(items):
        run = max(run, min(1.0, p_ * (m - j))); out[k] = run
    return out
prim10 = evaluate_g(TB, "MAE_60", "C3_hat", BASE10)
ok10 = bool(np.isfinite(prim10.get("clark_west_p", np.nan)) and prim10["clark_west_p"] < 0.05 and prim10["delta_r2_ci"][0] > 0)
sec10 = {f"MAE | {c}": evaluate_g(TB, "MAE_60", c, BASE10)
         for c in ["S60", "gain_multiple", "log_halflife", "term_reversion", "term_momentum", "C3_flow"]}
h10 = holm({k: v.get("clark_west_p", np.nan) for k, v in sec10.items()})
for k in sec10:
    sec10[k]["holm_p"] = h10[k]
sh = lambda v: {k: (round(v[k], 5) if isinstance(v.get(k), float) else v.get(k)) for k in
                ["n_train", "n_test", "n_symbols", "n_test_dates", "r2_base", "delta_r2", "delta_r2_ci", "delta_auc_ci",
                 "clark_west_t", "clark_west_p", "holm_p", "coef_candidates_train", "skipped"] if k in v}
print("=== PRIMARY (Bybit thin coins, real depth, beyond the Part 9 baseline):", "PASS" if ok10 else "FAIL"); print(sh(prim10))
for k, v in sec10.items():
    print(k, sh(v))
record("V6_bybit_ob_test", ok10, "real-data", {"primary": prim10, "secondary": sec10, "holm_p": h10, "symbols": SYMBOLS,
       "prereg_sha256": P10_HASH, "test_set_unlocks": n_unl10}, criterion=PREREG_P10["primary"]["success"])
zp10 = os.path.join(BASE, "p10_results.zip")
with zipfile.ZipFile(zp10, "w", zipfile.ZIP_DEFLATED) as z:
    for fp in glob.glob(os.path.join(RESULTS, "V*.json")) + [os.path.join(RESULTS, f) for f in
              ("p10_day_status.csv", "p10_trades_manifest_sha256.csv")] + [p10_path, pth10]:
        if os.path.exists(fp):
            z.write(fp, os.path.relpath(fp, BASE))
    for fp in glob.glob(os.path.join(OBMIN, "*", "*.json")):
        z.write(fp, os.path.relpath(fp, BASE))           # per-day book audit (sha256 of every order-book file)
print("zipped ->", zp10)
if IN_COLAB:
    from google.colab import files
    files.download(zp10)

Send `p10_results.zip` to Claude. If V0 says the trade archive is unreachable, or V4 includes fewer than 8 coins, tell Claude before running V6.